In [ ]:

#@title Cell 03.1 - Notebook overview
# This cell defines the purpose and fixed matching rule for Notebook 03.

from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 03: blaTEM-1 upper-MIC matched comparator selection

## Purpose

Notebook 03 selects three lower-MIC chromosomal comparators for each of the
16 upper-MIC blaTEM-1-only pathogens.

The selection uses the complete upper-MIC-to-comparison distance table from
Notebook 02.

## Fixed matching rule

For each upper-MIC pathogen:

1. require the comparator MIC to be at least 2 log2 units lower;
2. prioritize the smallest K-derived chromosomal distance;
3. when distances are equal, prefer the lower comparator MIC;
4. retain three comparators;
5. allow the same comparator to be used for more than one upper-MIC pathogen.

No genome differences are tested in this notebook.

## Notebook structure

Notebook 03 contains 7 code cells:

1. notebook overview;
2. locate the public repository and locate Notebook 02 outputs;
3. load and validate the matching inputs;
4. select three comparators for each upper-MIC pathogen;
5. assess matching quality and comparator reuse;
6. save the matched-pair and genome manifests;
7. save final QC, provenance and one complete output ZIP.
"""))

print(
    "Notebook 03 overview complete.\n"
    "Transition: Cell 03.2 will locate the public repository and locate the fixed "
    "Notebook 02 matching inputs."
)


In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()


#@title Cell 03.2 - Mount Drive and locate Notebook 02 inputs
# This cell defines the current-project inputs and Notebook 03 output paths.

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
PROJECT_ROOT = _repo_root()

# Notebook 01 empirical groups.
HIGH_MIC_PATH = (
    PROJECT_ROOT
    / '02_Data_New'
    / 'High_MIC_16'
    / '01_high_MIC_16_pathogens.csv'
)

COMPARISON_PATH = (
    PROJECT_ROOT
    / '02_Data_New'
    / 'Comparison_160'
    / '01_comparison_160_pathogens.csv'
)

# Notebook 02 fixed matching inputs.
ALL_DISTANCE_PATH = (
    PROJECT_ROOT
    / '04_Intermediate'
    / 'Relatedness'
    / '02_high_MIC_16_to_comparison_160_all_distances.csv.gz'
)

NEAREST10_PATH = (
    PROJECT_ROOT
    / '05_Results'
    / 'Tables'
    / '02_high_MIC_16_nearest_10_comparators.csv'
)

# Notebook 03 output directories.
MATCHED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / '02_Data_New'
    / 'Matched_Comparators'
)

RELATEDNESS_DIRECTORY = (
    PROJECT_ROOT
    / '04_Intermediate'
    / 'Relatedness'
)

TABLE_DIRECTORY = (
    PROJECT_ROOT
    / '05_Results'
    / 'Tables'
)

FIGURE_DIRECTORY = (
    PROJECT_ROOT
    / '05_Results'
    / 'Figures'
)

STATISTICAL_OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / '05_Results'
    / 'Statistical_Outputs'
)

for directory in [
    MATCHED_DATA_DIRECTORY,
    RELATEDNESS_DIRECTORY,
    TABLE_DIRECTORY,
    FIGURE_DIRECTORY,
    STATISTICAL_OUTPUT_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

required_inputs = [
    HIGH_MIC_PATH,
    COMPARISON_PATH,
    ALL_DISTANCE_PATH,
    NEAREST10_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        'Required input(s) were not found:\n'
        + '\n'.join(missing_inputs)
    )

print(f'Current project: {PROJECT_ROOT}')
print('All Notebook 03 fixed inputs were found.')
print(
    'Transition: Cell 03.3 will load and validate the 16 upper-MIC pathogens, '
    'the 160 comparison pathogens and all 2,560 pairwise distances.'
)


In [ ]:

#@title Cell 03.3 - Load and validate matching inputs
# This cell validates the empirical groups and the complete distance table
# created by Notebook 02.

high16 = pd.read_csv(HIGH_MIC_PATH)
comparison160 = pd.read_csv(COMPARISON_PATH)
all_distances = pd.read_csv(ALL_DISTANCE_PATH)
nearest10 = pd.read_csv(NEAREST10_PATH)

EXPECTED_HIGH_MIC = 16
EXPECTED_COMPARISON = 160
EXPECTED_PAIRS = EXPECTED_HIGH_MIC * EXPECTED_COMPARISON

if len(high16) != EXPECTED_HIGH_MIC:
    raise ValueError(
        f'Expected {EXPECTED_HIGH_MIC} upper-MIC pathogens; '
        f'found {len(high16)}.'
    )

if len(comparison160) != EXPECTED_COMPARISON:
    raise ValueError(
        f'Expected {EXPECTED_COMPARISON} comparison pathogens; '
        f'found {len(comparison160)}.'
    )

if len(all_distances) != EXPECTED_PAIRS:
    raise ValueError(
        f'Expected {EXPECTED_PAIRS} upper/comparison pairs; '
        f'found {len(all_distances)}.'
    )

required_distance_columns = [
    'upper_biosample',
    'upper_assembly_accession',
    'upper_log2_mic',
    'upper_mic_mg_L',
    'comparison_biosample',
    'comparison_assembly_accession',
    'comparison_log2_mic',
    'comparison_mic_mg_L',
    'delta_log2_mic_upper_minus_comparison',
    'K_relatedness',
    'K_derived_distance',
]

missing_columns = [
    column
    for column in required_distance_columns
    if column not in all_distances.columns
]

if missing_columns:
    raise ValueError(
        'Notebook 02 distance table is missing column(s): '
        + ', '.join(missing_columns)
    )

if high16['biosample'].duplicated().any():
    raise ValueError(
        'Upper-MIC table contains duplicate BioSamples.'
    )

if comparison160['biosample'].duplicated().any():
    raise ValueError(
        'Comparison table contains duplicate BioSamples.'
    )

if set(high16['biosample'].astype(str)) & set(
    comparison160['biosample'].astype(str)
):
    raise ValueError(
        'Upper-MIC and comparison groups overlap.'
    )

pair_counts = (
    all_distances
    .groupby('upper_biosample')
    .size()
)

if len(pair_counts) != EXPECTED_HIGH_MIC:
    raise ValueError(
        'Distance table does not contain all 16 upper-MIC pathogens.'
    )

if not (pair_counts == EXPECTED_COMPARISON).all():
    raise ValueError(
        'At least one upper-MIC pathogen does not have 160 comparison pairs.'
    )

numeric_columns = [
    'upper_log2_mic',
    'upper_mic_mg_L',
    'comparison_log2_mic',
    'comparison_mic_mg_L',
    'delta_log2_mic_upper_minus_comparison',
    'K_relatedness',
    'K_derived_distance',
]

for column in numeric_columns:
    values = pd.to_numeric(
        all_distances[column],
        errors='raise',
    ).to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            f'Column {column} contains a missing or non-finite value.'
        )

input_qc = pd.DataFrame([
    {'metric': 'Upper-MIC pathogens', 'value': len(high16)},
    {'metric': 'Comparison pathogens', 'value': len(comparison160)},
    {'metric': 'Upper/comparison pairs', 'value': len(all_distances)},
    {'metric': 'Pairs per upper-MIC pathogen', 'value': int(pair_counts.iloc[0])},
    {'metric': 'Notebook 02 nearest-10 rows', 'value': len(nearest10)},
])

display(input_qc)

print(
    'Input QC passed.\n'
    'Transition: Cell 03.4 will apply the fixed MIC-separation rule and '
    'select three chromosomally closest eligible comparators per pathogen.'
)


In [ ]:

#@title Cell 03.4 - Select three matched comparators per upper-MIC pathogen
# This cell applies the fixed matching rule:
# >=2 log2(MIC) lower, then minimum K-derived distance, with lower MIC used
# as the secondary ordering criterion when chromosomal distances are equal.

MINIMUM_LOG2_MIC_DIFFERENCE = 2.0
MATCHES_PER_UPPER = 3

eligible = all_distances.loc[
    all_distances[
        'delta_log2_mic_upper_minus_comparison'
    ] >= MINIMUM_LOG2_MIC_DIFFERENCE
].copy()

eligible_counts = (
    eligible
    .groupby('upper_biosample')
    .size()
)

if len(eligible_counts) != EXPECTED_HIGH_MIC:
    raise ValueError(
        'At least one upper-MIC pathogen has no eligible comparator.'
    )

if (eligible_counts < MATCHES_PER_UPPER).any():
    failed = eligible_counts.loc[
        eligible_counts < MATCHES_PER_UPPER
    ]

    raise ValueError(
        'At least one upper-MIC pathogen has fewer than three eligible '
        f'comparators:\n{failed}'
    )

eligible = (
    eligible
    .sort_values(
        [
            'upper_biosample',
            'K_derived_distance',
            'comparison_mic_mg_L',
            'comparison_biosample',
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

eligible['eligible_distance_rank'] = (
    eligible
    .groupby('upper_biosample')
    .cumcount()
    + 1
)

selected_pairs = (
    eligible
    .groupby(
        'upper_biosample',
        sort=False,
        group_keys=False,
    )
    .head(MATCHES_PER_UPPER)
    .copy()
)

selected_pairs['match_rank'] = (
    selected_pairs
    .groupby('upper_biosample')
    .cumcount()
    + 1
)

selected_pairs = selected_pairs[
    [
        'upper_biosample',
        'upper_assembly_accession',
        'upper_log2_mic',
        'upper_mic_mg_L',
        'match_rank',
        'comparison_biosample',
        'comparison_assembly_accession',
        'comparison_log2_mic',
        'comparison_mic_mg_L',
        'delta_log2_mic_upper_minus_comparison',
        'K_relatedness',
        'K_derived_distance',
        'eligible_distance_rank',
    ]
].reset_index(drop=True)

expected_selected_rows = (
    EXPECTED_HIGH_MIC
    * MATCHES_PER_UPPER
)

if len(selected_pairs) != expected_selected_rows:
    raise ValueError(
        f'Expected {expected_selected_rows} selected matched pairs; '
        f'found {len(selected_pairs)}.'
    )

selected_counts = (
    selected_pairs
    .groupby('upper_biosample')
    .size()
)

if not (selected_counts == MATCHES_PER_UPPER).all():
    raise ValueError(
        'At least one upper-MIC pathogen does not have exactly three matches.'
    )

if (
    selected_pairs[
        'delta_log2_mic_upper_minus_comparison'
    ] < MINIMUM_LOG2_MIC_DIFFERENCE
).any():
    raise ValueError(
        'At least one selected comparator does not meet the MIC-separation rule.'
    )

print('Selected matched pairs:')
display(selected_pairs)

print(
    'Transition: Cell 03.5 will assess chromosomal match quality, '
    'MIC separation and repeated use of the same comparator.'
)


In [ ]:

#@title Cell 03.5 - Assess matching quality and comparator reuse
# This cell summarizes the 48 selected pairs and quantifies how much
# chromosomal distance was added by requiring a >=2 log2(MIC) difference.

per_upper_rows = []

for upper_biosample, selected_group in selected_pairs.groupby(
    'upper_biosample'
):
    selected_group = selected_group.sort_values('match_rank')

    all_group = (
        all_distances.loc[
            all_distances['upper_biosample'] == upper_biosample
        ]
        .sort_values(
            [
                'K_derived_distance',
                'comparison_mic_mg_L',
                'comparison_biosample',
            ]
        )
        .reset_index(drop=True)
    )

    closest_three_unrestricted = (
        all_group
        .head(MATCHES_PER_UPPER)
    )

    selected_distances = selected_group[
        'K_derived_distance'
    ].to_numpy(dtype=float)

    selected_delta = selected_group[
        'delta_log2_mic_upper_minus_comparison'
    ].to_numpy(dtype=float)

    unrestricted_distances = closest_three_unrestricted[
        'K_derived_distance'
    ].to_numpy(dtype=float)

    per_upper_rows.append({
        'upper_biosample': upper_biosample,
        'upper_log2_mic': float(
            selected_group['upper_log2_mic'].iloc[0]
        ),
        'upper_mic_mg_L': float(
            selected_group['upper_mic_mg_L'].iloc[0]
        ),
        'minimum_selected_distance': float(
            selected_distances.min()
        ),
        'median_selected_distance': float(
            np.median(selected_distances)
        ),
        'maximum_selected_distance': float(
            selected_distances.max()
        ),
        'minimum_selected_delta_log2_mic': float(
            selected_delta.min()
        ),
        'median_selected_delta_log2_mic': float(
            np.median(selected_delta)
        ),
        'maximum_selected_delta_log2_mic': float(
            selected_delta.max()
        ),
        'median_unrestricted_closest3_distance': float(
            np.median(unrestricted_distances)
        ),
        'distance_penalty_median': float(
            np.median(selected_distances)
            - np.median(unrestricted_distances)
        ),
    })

per_upper_match_quality = (
    pd.DataFrame(per_upper_rows)
    .sort_values(
        [
            'upper_mic_mg_L',
            'upper_biosample',
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

comparator_reuse = (
    selected_pairs
    .groupby(
        [
            'comparison_biosample',
            'comparison_assembly_accession',
            'comparison_log2_mic',
            'comparison_mic_mg_L',
        ],
        dropna=False,
    )
    .agg(
        times_selected=(
            'upper_biosample',
            'size',
        ),
        distinct_upper_pathogens=(
            'upper_biosample',
            'nunique',
        ),
        minimum_K_derived_distance=(
            'K_derived_distance',
            'min',
        ),
        median_K_derived_distance=(
            'K_derived_distance',
            'median',
        ),
        maximum_K_derived_distance=(
            'K_derived_distance',
            'max',
        ),
    )
    .reset_index()
    .sort_values(
        [
            'distinct_upper_pathogens',
            'times_selected',
            'median_K_derived_distance',
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

overall_match_summary = pd.DataFrame([
    {
        'metric': 'Upper-MIC pathogens matched',
        'value': EXPECTED_HIGH_MIC,
    },
    {
        'metric': 'Comparators per upper-MIC pathogen',
        'value': MATCHES_PER_UPPER,
    },
    {
        'metric': 'Selected matched pairs',
        'value': len(selected_pairs),
    },
    {
        'metric': 'Minimum required delta log2(MIC)',
        'value': MINIMUM_LOG2_MIC_DIFFERENCE,
    },
    {
        'metric': 'Unique selected comparison pathogens',
        'value': int(
            selected_pairs[
                'comparison_biosample'
            ].nunique()
        ),
    },
    {
        'metric': 'Maximum number of upper-MIC pathogens sharing one comparator',
        'value': int(
            comparator_reuse[
                'distinct_upper_pathogens'
            ].max()
        ),
    },
    {
        'metric': 'Median selected K-derived distance',
        'value': float(
            selected_pairs[
                'K_derived_distance'
            ].median()
        ),
    },
    {
        'metric': 'Median selected delta log2(MIC)',
        'value': float(
            selected_pairs[
                'delta_log2_mic_upper_minus_comparison'
            ].median()
        ),
    },
])

PER_UPPER_MATCH_QUALITY_PATH = (
    TABLE_DIRECTORY
    / '03_match_quality_by_upper_MIC_pathogen.csv'
)

COMPARATOR_REUSE_PATH = (
    TABLE_DIRECTORY
    / '03_selected_comparator_reuse.csv'
)

OVERALL_MATCH_SUMMARY_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '03_matching_summary.csv'
)

per_upper_match_quality.to_csv(
    PER_UPPER_MATCH_QUALITY_PATH,
    index=False,
)

comparator_reuse.to_csv(
    COMPARATOR_REUSE_PATH,
    index=False,
)

overall_match_summary.to_csv(
    OVERALL_MATCH_SUMMARY_PATH,
    index=False,
)

print('Match quality by upper-MIC pathogen:')
display(per_upper_match_quality)

print('\nOverall matching summary:')
display(overall_match_summary)

print('\nComparator reuse:')
display(comparator_reuse)

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    selected_pairs['K_derived_distance'],
    selected_pairs[
        'delta_log2_mic_upper_minus_comparison'
    ],
)

ax.axhline(
    MINIMUM_LOG2_MIC_DIFFERENCE,
    linewidth=1,
    linestyle='--',
)

ax.set_xlabel(
    'K-derived chromosomal distance'
)

ax.set_ylabel(
    'Upper-MIC minus comparator log2(MIC)'
)

ax.set_title(
    'Selected matched comparisons'
)

fig.tight_layout()

MATCH_QUALITY_FIGURE_PATH = (
    FIGURE_DIRECTORY
    / '03_selected_matches_distance_vs_MIC_difference.png'
)

fig.savefig(
    MATCH_QUALITY_FIGURE_PATH,
    dpi=300,
    bbox_inches='tight',
)

plt.show()

print(f'Saved: {PER_UPPER_MATCH_QUALITY_PATH}')
print(f'Saved: {COMPARATOR_REUSE_PATH}')
print(f'Saved: {OVERALL_MATCH_SUMMARY_PATH}')
print(f'Saved: {MATCH_QUALITY_FIGURE_PATH}')

print(
    '\nTransition: Cell 03.6 will save the 48 matched pairs, '
    'the unique selected comparators and the genome manifest for '
    'the subsequent direct genome-comparison analysis.'
)


In [ ]:

#@title Cell 03.6 - Save matched-pair and genome manifests
# This cell creates the fixed matched comparison set for the next notebook.
# Reused comparators appear once in the unique-comparator and genome manifests.

MATCHED_PAIRS_PATH = (
    MATCHED_DATA_DIRECTORY
    / '03_matched_pairs_16x3.csv'
)

SELECTED_COMPARATORS_PATH = (
    MATCHED_DATA_DIRECTORY
    / '03_selected_comparator_pathogens.csv'
)

GENOME_MANIFEST_PATH = (
    MATCHED_DATA_DIRECTORY
    / '03_genome_comparison_manifest.csv'
)

selected_pairs.to_csv(
    MATCHED_PAIRS_PATH,
    index=False,
)

selected_comparator_ids = (
    selected_pairs[
        'comparison_biosample'
    ]
    .astype(str)
    .unique()
    .tolist()
)

comparison_lookup = (
    comparison160
    .assign(
        biosample=lambda x: x[
            'biosample'
        ].astype(str)
    )
    .set_index('biosample')
)

selected_comparators = (
    comparison_lookup
    .loc[selected_comparator_ids]
    .reset_index()
    .copy()
)

reuse_count_map = (
    selected_pairs
    .groupby('comparison_biosample')[
        'upper_biosample'
    ]
    .nunique()
    .to_dict()
)

selected_comparators[
    'n_upper_MIC_matches'
] = (
    selected_comparators[
        'biosample'
    ]
    .map(reuse_count_map)
    .astype(int)
)

selected_comparators.to_csv(
    SELECTED_COMPARATORS_PATH,
    index=False,
)

high_manifest = high16.copy()

high_manifest['biosample'] = (
    high_manifest[
        'biosample'
    ].astype(str)
)

high_manifest = high_manifest[
    [
        'biosample',
        'assembly_accession',
        'log2_mic',
        'observed_mic',
    ]
].copy()

high_manifest['comparison_role'] = (
    'upper_MIC'
)

high_manifest['n_upper_MIC_matches'] = np.nan

comparator_manifest = selected_comparators[
    [
        'biosample',
        'assembly_accession',
        'log2_mic',
        'observed_mic',
        'n_upper_MIC_matches',
    ]
].copy()

comparator_manifest['comparison_role'] = (
    'matched_comparator'
)

genome_manifest = pd.concat(
    [
        high_manifest,
        comparator_manifest,
    ],
    ignore_index=True,
    sort=False,
)

genome_manifest = genome_manifest[
    [
        'comparison_role',
        'biosample',
        'assembly_accession',
        'log2_mic',
        'observed_mic',
        'n_upper_MIC_matches',
    ]
]

if genome_manifest['biosample'].duplicated().any():
    raise ValueError(
        'Genome comparison manifest contains duplicate BioSamples.'
    )

if genome_manifest['assembly_accession'].isna().any():
    raise ValueError(
        'At least one selected genome lacks an assembly accession.'
    )

genome_manifest.to_csv(
    GENOME_MANIFEST_PATH,
    index=False,
)

manifest_summary = pd.DataFrame([
    {
        'metric': 'Upper-MIC genomes',
        'value': int(
            (
                genome_manifest[
                    'comparison_role'
                ] == 'upper_MIC'
            ).sum()
        ),
    },
    {
        'metric': 'Unique matched comparator genomes',
        'value': int(
            (
                genome_manifest[
                    'comparison_role'
                ] == 'matched_comparator'
            ).sum()
        ),
    },
    {
        'metric': 'Total unique genomes for direct comparison',
        'value': len(genome_manifest),
    },
    {
        'metric': 'Matched pair rows',
        'value': len(selected_pairs),
    },
])

print('Genome-comparison manifest summary:')
display(manifest_summary)

print('\nMatched pairs:')
display(selected_pairs)

print(f'\nSaved: {MATCHED_PAIRS_PATH}')
print(f'Saved: {SELECTED_COMPARATORS_PATH}')
print(f'Saved: {GENOME_MANIFEST_PATH}')

print(
    '\nTransition: Cell 03.7 will save final QC, provenance and one complete '
    'Notebook 03 output ZIP.'
)


In [ ]:

#@title Cell 03.7 - Save final QC, provenance and Notebook 03 output ZIP
# This cell records the fixed matching rule, validates the final comparison
# set and saves one complete Notebook 03 output ZIP.

QC_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '03_qc_summary.csv'
)

MANIFEST_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '03_manifest.json'
)

FINAL_ZIP_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '03_TEM1_High_MIC_Matched_Comparator_Selection_outputs.zip'
)

final_qc = pd.DataFrame([
    {
        'metric': 'Upper-MIC pathogens',
        'value': len(high16),
    },
    {
        'metric': 'Comparison pathogens available',
        'value': len(comparison160),
    },
    {
        'metric': 'Minimum delta log2(MIC)',
        'value': MINIMUM_LOG2_MIC_DIFFERENCE,
    },
    {
        'metric': 'Comparators selected per upper-MIC pathogen',
        'value': MATCHES_PER_UPPER,
    },
    {
        'metric': 'Matched pair rows',
        'value': len(selected_pairs),
    },
    {
        'metric': 'Unique selected comparator genomes',
        'value': selected_pairs[
            'comparison_biosample'
        ].nunique(),
    },
    {
        'metric': 'Total unique genomes in comparison manifest',
        'value': len(genome_manifest),
    },
    {
        'metric': 'All selected pairs satisfy MIC rule',
        'value': bool(
            (
                selected_pairs[
                    'delta_log2_mic_upper_minus_comparison'
                ] >= MINIMUM_LOG2_MIC_DIFFERENCE
            ).all()
        ),
    },
    {
        'metric': 'Exactly three matches per upper-MIC pathogen',
        'value': bool(
            (
                selected_pairs
                .groupby('upper_biosample')
                .size()
                == MATCHES_PER_UPPER
            ).all()
        ),
    },
])

final_qc.to_csv(
    QC_PATH,
    index=False,
)

manifest = {
    'notebook': (
        '03_TEM1_High_MIC_Matched_Comparator_Selection.ipynb'
    ),
    'project': 'Ceftazidime_Chromosomal_Evolution',
    'purpose': (
        'Select three chromosomally close lower-MIC comparators '
        'for each of the 16 upper-MIC blaTEM-1-only pathogens.'
    ),
    'inputs': {
        'upper_MIC_16': str(HIGH_MIC_PATH),
        'comparison_160': str(COMPARISON_PATH),
        'all_pairwise_distances': str(ALL_DISTANCE_PATH),
        'nearest_10_table': str(NEAREST10_PATH),
    },
    'matching_rule': {
        'minimum_delta_log2_MIC': (
            MINIMUM_LOG2_MIC_DIFFERENCE
        ),
        'comparators_per_upper_MIC_pathogen': (
            MATCHES_PER_UPPER
        ),
        'primary_priority': (
            'minimum K-derived chromosomal distance'
        ),
        'secondary_priority': (
            'lower comparator MIC when chromosomal distances are equal'
        ),
        'comparator_reuse_allowed': True,
    },
    'direct_genome_comparison_status': (
        'not started; Notebook 03 fixes the matched comparison set only'
    ),
}

with open(
    MANIFEST_PATH,
    'w',
    encoding='utf-8',
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

output_files = [
    MATCHED_PAIRS_PATH,
    SELECTED_COMPARATORS_PATH,
    GENOME_MANIFEST_PATH,
    PER_UPPER_MATCH_QUALITY_PATH,
    COMPARATOR_REUSE_PATH,
    OVERALL_MATCH_SUMMARY_PATH,
    MATCH_QUALITY_FIGURE_PATH,
    QC_PATH,
    MANIFEST_PATH,
]

for path in output_files:
    if not path.exists():
        raise FileNotFoundError(
            f'Expected Notebook 03 output was not created: {path}'
        )

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in output_files:
        archive.write(
            path,
            arcname=str(
                path.relative_to(PROJECT_ROOT)
            ),
        )

print('Final QC:')
display(final_qc)

print(f'\nSaved: {QC_PATH}')
print(f'Saved: {MANIFEST_PATH}')
print(f'Saved: {FINAL_ZIP_PATH}')

print(
    '\nNotebook 03 complete.\n'
    'The matched comparison set is now fixed for the subsequent '
    'direct genome-comparison analysis.'
)
